## Library Installs & Imports

In [0]:
%pip install rouge-score -q
%pip install -U mlflow[genai] -q
%pip install databricks-agents -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import pyspark.sql.functions as F
import mlflow
from mlflow.metrics.genai import make_genai_metric, EvaluationExample
import json

## Create an example dataset with a few randomly selected DAIS Abstracts

In [0]:
eval_sessions = pd.Series([
  """In Bruce’s career in cyber warfare and enterprise cybersecurity, he worked on many of the highest profile botnet and nation state takedowns in history. He also helped build the tech in one of the world’s most advanced SOCs.

 

Bruce will explain what he learned from that experience and why it prompted him to leave early retirement, sell his beloved sports car and co-found ziggiz.

 

We all know there’s more data than ever. Anyone close to cybersecurity also knows that SIEMs, typically at the center of enterprise cybersecurity operations, have become too expensive even at the highest levels of government and Fortune 100s.""",
"""DLT 's home page says, “It’s a declarative ETL framework (...) that helps data teams simplify streaming and batch ETL cost-effectively. Simply define the transformations to perform on your data and let DLT pipelines automatically manage task orchestration, cluster management, monitoring, data quality and error handling.

 

This talk aims to show you how DLT saved me a lot of trouble while on a tight delivery schedule. I’ll show you why the DLT headline is correct. In other words, I hope I will convince you to consider the DLT framework for your next ETL project.

 

I found over 10 reasons why investing in DLT for your next project is worth your time.

 

I will discuss the foundational concepts (Spark SQL and Structured Streaming, Delta Lake) and more importantly, how they paved the way for DLT.

 

The talk is based on my recent experience with two successful projects, which have done very well from their humble beginnings and were so much fun to be part of.""",
"""Josue is well known for his practical perspectives on the data and AI landscape. We'll talk about what he is seeing in the market, his take on product feature updates, and some humor mixed in.""",
"""SMBC, a major Japanese multinational financial services institution, has embarked on an initiative to build a GenAI-powered, modern and well-governed cloud data platform on Azure/Databricks. This initiative aims to build an enterprise data foundation encompassing loans, deposits, securities, derivatives, and other data domains.

Its primary goals are:

To decommission legacy data platforms and reduce data sprawl by migrating 20+ core banking systems to a multi-tenant Azure Databricks architecture
To leverage Databrick’s delta-share capabilities to address SMBC’s unique global footprint and data sharing needs
To govern data by design using Unity Catalog
To achieve global adoption of the frameworks, accelerators, architecture and tool stack to support similar implementations across EMEA
Deloitte and SMBC leveraged the Brickbuilder asset “Data as a Service for Banking” to accelerate this highly strategic transformation.""",
"""Deploying AI in production is getting more complex — with different model types, tighter timelines, and growing infrastructure demands. In this session, we’ll walk through how Mosaic AI Model Serving helps teams deploy and scale both traditional ML and generative AI models efficiently, with built-in monitoring and governance.We’ll also hear from Skyscanner on how they’ve integrated AI into their products, scaled to 100+ production endpoints, and built the processes and team structures to support AI at scale.

 

Key Takeaways:

How Skyscanner ships and operates AI in real-world products
How to deploy and scale a variety of models with low latency and minimal overhead
Building compound AI systems using models, feature stores, and vector search
Monitoring, debugging, and governing production workloads""",
"""We present AT&T AutoClassify, built jointly between AT&T's Chief Data Office (CDO) and Databricks professional services, a novel end-to-end system for automatic multi-head binary classifications from unlabeled text data. Our approach automates the challenge of creating labeled datasets and training multi-head binary classifiers with minimal human intervention.

 

Starting only from a corpus of unlabeled text and a list of desired labels, AT&T AutoClassify leverages advanced natural language processing techniques to automatically mine relevant examples from raw text, fine-tune embedding models and train individual classifier heads for multiple true/false labels. This solution can reduce LLM classification costs by 1,000x, making it an efficient solution in operational costs.

 

The end result is a highly optimized and low-cost model servable in Databricks capable of taking raw text and producing multiple binary classifications. An example use case using call transcripts will be examined.""",
"""Enterprises generate massive amounts of unstructured data — from support tickets and PDFs to emails and product images. But extracting insight from that data requires brittle pipelines and complex tools.

 

Databricks AI Functions make this simpler. In this session, you’ll learn how to apply powerful language and vision models directly within your SQL and ETL workflows — no endpoints, no infrastructure, no rewrites.

 

We’ll explore practical use cases and best practices for analyzing complex documents, classifying issues, translating content, and inspecting images — all in a way that’s scalable, declarative, and secure.

 

What you’ll learn:

How to run state-of-the-art LLMs like GPT-4, Claude Sonnet 4, and Llama 4 on your data
How to build scalable, multimodal ETL workflows for text and images
Best practices for prompts, cost, and error handling in production
Real-world examples of GenAI use cases powered by AI Functions"""
])

eval_sessions_df = pd.DataFrame({"session_desc" : eval_sessions})

Create a temporary view with the example data.

In [0]:
(
  spark
  .createDataFrame(eval_sessions_df)
  .createOrReplaceTempView("dais_sessions")
)

# Use Case 1:  Summaries of DAIS Abstracts
_Why?_ These will be used for automated social media promotion posts.

Use the `ai_summarize` task function to easily generate the summaries.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW session_summaries AS
SELECT 
  session_desc, 
  ai_summarize(session_desc) as summary 
FROM dais_sessions

In [0]:
# Print out the summaries
display(spark.sql("SELECT summary FROM session_summaries"))

summary
"Bruce, a cyber warfare expert, shares his experience with botnet takedowns and SOCs, leading him to co-found ziggiz due to expensive SIEMs."
"DLT is a declarative ETL framework simplifying streaming and batch data processing, saving time and trouble in data projects."
"Josue shares practical insights on data and AI, discussing market trends and product updates with humor."
"SMBC builds a GenAI-powered cloud data platform on Azure/Databricks to modernize and govern its data, migrating 20+ core systems and leveraging delta-share capabilities."
"Deploying AI models efficiently with Mosaic AI, featuring Skyscanner's success story and key takeaways on scaling AI production."
"AT&T AutoClassify: an automated system for multi-head binary classifications from unlabeled text data, reducing costs by 1,000x."
Databricks AI Functions simplify extracting insights from unstructured data using powerful language and vision models within SQL and ETL workflows.


The summaries look great! Let's get a feel for a summarization metric.

Test using the RougeL metric. While ROUGE can be a good measure of summaries in some regard, it is only testing for overlapping n-grams. It heavily favors extractive summaries and not rephrasings. Let's assume for this use case we've determined RougeL is a good metric for us to use.

Read the example data into a pandas dataframe and use mlflow to evaluate and generate aggregate metrics.

In [0]:
evaluation_set = spark.sql("SELECT * FROM session_summaries").toPandas()
evaluation_set

,session_desc,summary
0,In Bruce’s career in cyber warfare and enterpr...,"Bruce, a cyber warfare expert, shares his expe..."
1,"DLT 's home page says, “It’s a declarative ETL...",DLT is a declarative ETL framework simplifying...
2,Josue is well known for his practical perspect...,Josue shares practical insights on data and AI...
3,"SMBC, a major Japanese multinational financial...",SMBC builds a GenAI-powered cloud data platfor...
4,Deploying AI in production is getting more com...,Deploying AI models efficiently with Mosaic AI...
5,"We present AT&T AutoClassify, built jointly be...",AT&T AutoClassify: an end-to-end system for au...
6,Enterprises generate massive amounts of unstru...,Databricks AI Functions simplify extracting in...


In [0]:
from rouge_score import rouge_scorer
from mlflow.genai.scorers import scorer
from typing import Optional, Any
from mlflow.entities import Feedback


# Create a data structure that migrates from mlflow 2 to mlflow 3
eval_data = [
    {
        "inputs": {"session_desc": row["session_desc"]},
        "outputs": {"summary": row["summary"]}
    }
    for _, row in evaluation_set.iterrows()
]

@scorer(aggregations=["mean", "variance"])
def rougeL_scorer(
  inputs: Optional[dict[str, Any]],  # The agent's raw input, parsed from the Trace or dataset, as a Python dict
  outputs: Optional[Any],  # The agent's raw output, parsed from the Trace or
):
    candidate = outputs.get("summary", "")
    reference = inputs.get("session_desc", "")
    scorer_ = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    scores = scorer_.score(reference, candidate)
    return scores['rougeL'].fmeasure

# Evaluate with the ROUGEL score
results = mlflow.genai.evaluate(
      data=eval_data,
      predict_fn=None,
      scorers=[rougeL_scorer],
  )

print(results.metrics)

Evaluating:   0%|          | 0/7 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

{'agent/latency_seconds/mean': 0.0, 'rougeL_scorer/mean': 0.2304115209459119}


[Trace(trace_id=tr-cf732662c572c3f3175ddab3b4f641bd), Trace(trace_id=tr-59ae952f27f75cf8765103880574e5e5), Trace(trace_id=tr-b3d3aafd967b7a83c1aad185beb241ab), Trace(trace_id=tr-1723b6a6d45b8043bd787112aa806ce5), Trace(trace_id=tr-4bdc65ae3792b44aaf0744278e460e45), Trace(trace_id=tr-0a4087daade84d9d0fbe5a1849510ffb), Trace(trace_id=tr-00389d83c80226fe5cd02f9aac342d28)]

Maybe we can do a bit better using a tailored prompt?

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW session_summaries_tailored AS
SELECT 
  session_desc, 
  ai_query(
    "databricks-meta-llama-3-3-70b-instruct",
    "Summarize the following Data & AI Summit abstract. Ensure you include all key technologies, entities, names, and keywords. The summary will be used for a social media promotion post. Abstract: " || session_desc
  ) as summary 
FROM dais_sessions

In [0]:
evaluation_set = spark.sql("SELECT * FROM session_summaries_tailored").toPandas()

# Create a data structure that migrates from mlflow 2 to mlflow 3
eval_data = [
    {
        "inputs": {"session_desc": row["session_desc"]},
        "outputs": {"summary": row["summary"]}
    }
    for _, row in evaluation_set.iterrows()
]

# Evaluate with the ROUGEL score
results = mlflow.genai.evaluate(
      data=eval_data,
      predict_fn=None,
      scorers=[rougeL_scorer],
  )

print(results.metrics)

Evaluating:   0%|          | 0/7 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

{'agent/latency_seconds/mean': 0.0, 'rougeL_scorer/mean': 0.3852955618179256}


[Trace(trace_id=tr-b74b6bde6533a25a5cc4c36cf44fd175), Trace(trace_id=tr-94e70d63789468304c7f9728c3c837d9), Trace(trace_id=tr-8d8d3411f4fc11ede00efccc790077ae), Trace(trace_id=tr-4efe8b04ccad9d00a22f5e2d8e84e442), Trace(trace_id=tr-d5ec8bd124399b43e75728498c54edb4), Trace(trace_id=tr-06a331cda7d4a7de548bf1636fecd795), Trace(trace_id=tr-440ac61b946a0106cb1ce4abcff2fc5c)]

🎉🎉🎉 Awesome! Metrics have improved! 🎉🎉🎉

In [0]:
# Print out the summaries
display(spark.sql("SELECT summary FROM session_summaries_tailored"))

summary
"Here's a summary of the Data & AI Summit abstract for a social media promotion post: ""Hear from Bruce, a veteran of cyber warfare and enterprise cybersecurity, who's worked on historic botnet and nation-state takedowns! He'll share his expertise on the limitations of traditional Security Information and Event Management (SIEM) systems, even for top gov't agencies and Fortune 100s. Learn how his experience led him to co-found #ziggiz and tackle the challenges of overwhelming data in cybersecurity. Join us at the Data & AI Summit to discover the future of cybersecurity! #cybersecurity #SIEM #DataAI #ziggiz"""
"Here's a social media promotion post summarizing the Data & AI Summit abstract: ""Join us at the Data & AI Summit to learn how DLT, a declarative ETL framework, can simplify your streaming and batch ETL processes! Discover how DLT's automated task orchestration, cluster management, monitoring, data quality, and error handling can save you time and trouble. Hear from a speaker who's used DLT on two successful projects, leveraging foundational concepts like Spark SQL, Structured Streaming, and Delta Lake. Find out why they believe DLT is worth considering for your next ETL project and learn about the 10+ reasons why it's a game-changer! #DataAISummit #DLT #ETL #SparkSQL #StructuredStreaming #DeltaLake #DataEngineering #AI"""
"Get ready for insights from Josue, a renowned expert in the data and AI landscape! Join us at the Data & AI Summit to hear his practical perspectives on the latest market trends, product feature updates, and more! With a dash of humor, Josue will share his take on the current state of #DataScience, #ArtificialIntelligence, and #MachineLearning. Don't miss this opportunity to learn from a thought leader in the industry! #DataAIsummit #Josue #DataAndAI #Innovation"
"Here's a summary of the abstract for a social media promotion post: ""Exciting news from the world of finance and tech! SMBC, a leading Japanese multinational financial services institution, is revolutionizing its data platform with a GenAI-powered, cloud-based solution on Azure and Databricks! In partnership with Deloitte, SMBC is migrating 20+ core banking systems to a multi-tenant Azure Databricks architecture, leveraging Databricks' delta-share capabilities and Unity Catalog for data governance. This strategic initiative aims to: Decommission legacy data platforms and reduce data sprawl Enable global data sharing and adoption across EMEA Accelerate transformation with Brickbuilder's ""Data as a Service for Banking"" asset Stay tuned for more updates on this innovative project that's set to transform the banking industry! #GenAI #Azure #Databricks #DataGovernance #CloudDataPlatform #Banking #FinancialServices #DigitalTransformation #Deloitte #SMBC"""
"Here's a social media promotion post summarizing the Data & AI Summit abstract: ""Join us at the Data & AI Summit to learn how to deploy and scale AI models efficiently! Hear from Skyscanner on how they've successfully integrated AI into their products, scaling to 100+ production endpoints. Discover how Mosaic AI Model Serving can help you deploy traditional ML and generative AI models with built-in monitoring and governance. Key takeaways include: Shipping and operating AI in real-world products with Skyscanner Deploying and scaling models with low latency and minimal overhead Building compound AI systems using models, feature stores, and vector search Monitoring, debugging, and governing production workloads Don't miss this opportunity to learn from industry experts and stay ahead of the curve in AI deployment! #DataAISummit #AI #MachineLearning #MosaicAI #Skyscanner #ModelServing #GenerativeAI #ML"""
"Here's a social media promotion post summarizing the Data & AI Summit abstract: ""Introducing AT&T AutoClassify, a game-changing solution built in collaboration with @Databricks! This innovative system uses advanced #NaturalLanguageProcessing (NLP) techniques to automate multi-head b

These look even better but we're going to have to manually parse out the extraneous text.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW session_summaries_tailored_structured AS
SELECT 
  session_desc, 
  ai_query(
    "databricks-meta-llama-3-3-70b-instruct",
    "Summarize the following Data & AI Summit abstract. Ensure you include all key technologies, entities, names, and keywords. The summary will be used for a social media promotion post. Abstract: " || session_desc,
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "social_media_summary_post",
        "schema": {
          "type": "object",
          "properties": {
            "social_media_summary": {"type": "string"}
          }
      },
      "strict": true
    }
  }'
  ) as summary 
FROM dais_sessions

In [0]:
display(spark.sql("SELECT summary FROM session_summaries_tailored_structured"))

summary
"{""social_media_summary"": ""Join us at the Data & AI Summit to hear from Bruce, a veteran of cyber warfare and enterprise cybersecurity, who worked on high-profile botnet and nation-state takedowns and helped build advanced Security Operations Centers (SOCs). He'll share his experience and why it led him to co-found ziggiz. With the surge in data and the limitations of traditional Security Information and Event Management (SIEM) systems, even for top governments and Fortune 100s, Bruce's insights will shed light on the future of cybersecurity. #DataAISummit #Cybersecurity #SIEM #SOC #Botnet #NationState #Ziggiz""}"
"{""social_media_summary"": ""Discover how DLT, a declarative ETL framework, simplifies streaming and batch ETL cost-effectively! Learn how DLT pipelines automate task orchestration, cluster management, monitoring, data quality, and error handling. Join us to explore the foundational concepts of Spark SQL, Structured Streaming, and Delta Lake, and find out why DLT is a game-changer for your next ETL project. Hear from a speaker who's worked on two successful projects and get convinced to consider DLT for your next project! #DLT #ETL #SparkSQL #StructuredStreaming #DeltaLake #DataAI #DataTeams""}"
"{""social_media_summary"": ""Join us at the Data & AI Summit as we sit down with Josue, a renowned expert in the data and AI landscape, to discuss his practical perspectives on market trends, product feature updates, and more, all with a dash of humor! #DataAISummit #AI #DataLandscape #Josue""}"
"{""social_media_summary"": ""Discover how SMBC, a leading Japanese financial services institution, is revolutionizing its data platform with GenAI-powered Azure/Databricks! With the help of Deloitte, they're migrating 20+ core banking systems to a multi-tenant architecture, leveraging Databricks' delta-share capabilities, and governing data with Unity Catalog. This strategic transformation aims to reduce data sprawl, achieve global adoption, and support similar implementations across EMEA. #GenAI #Azure #Databricks #DataAsAService #Banking #DigitalTransformation #CloudDataPlatform #DataGovernance #Deloitte #SMBC""}"
"{""social_media_summary"": ""Don't miss our upcoming session at the Data & AI Summit! Learn how Mosaic AI Model Serving helps teams deploy & scale traditional ML & generative AI models efficiently. Hear from Skyscanner on how they've integrated AI into their products, scaled to 100+ production endpoints, and built processes to support AI at scale. Key takeaways include: shipping & operating AI in real-world products, deploying & scaling models with low latency, building compound AI systems using models, feature stores, & vector search, and monitoring, debugging, & governing production workloads. #AI #ML #DataAI #MosaicAI #Skyscanner #AIatScale""}"
"{""social_media_summary"": ""Introducing AT&T AutoClassify, a groundbreaking end-to-end system for automatic multi-head binary classifications from unlabeled text data! Developed jointly by AT&T's Chief Data Office (CDO) and Databricks professional services, this innovative solution leverages advanced natural language processing (NLP) techniques to automate the creation of labeled datasets and train multi-head binary classifiers with minimal human intervention. With the potential to reduce LLM classification costs by 1,000x, AT&T AutoClassify is a game-changer for operational efficiency. Join us to explore an example use case using call transcripts and discover how this highly optimized and low-cost model, servable in Databricks, can revolutionize your text classification tasks! #ATTAutoClassify #Databricks #NLP #LLM #TextClassification #AI #MachineLearning""}"
"{""social_media_summary"": ""Unlock insights from unstructured data with Databricks AI Functions! Learn how to apply powerful language & vision models directly within SQL & ETL workflows, no infrastructure needed. Discover practical use cases for analyzing complex documents, classifying issues, translating

# Use Case 2: Generate titles and keywords for DAIS sessions.
_Why?_  We want to create the catchiest titles to grab the attention of DAIS attendees.

Let's use `ai_query` to generate catchy titles for our presentations and some relevant keywords.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW generated_titles_keywords AS
SELECT 
  session_desc,
  ai_query(
    "databricks-meta-llama-3-3-70b-instruct",
    "Given a description of a DAIS 2025 session, generate an catchy talk title and choose 3 keywords: " || session_desc,
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "research_paper_extraction",
        "schema": {
          "type": "object",
          "properties": {
            "generated_title": {"type": "string"},
            "keyword_1": {"type": "string"},
            "keyword_2": {"type": "string"},
            "keyword_3": {"type": "string"}
          }
      },
      "strict": true
    }
  }'
  ) as extracted_info
FROM dais_sessions

In [0]:
%sql
SELECT extracted_info FROM generated_titles_keywords

extracted_info
"{""generated_title"": ""From Botnet Battles to Revolutionizing SIEM: Lessons from the Frontlines"", ""keyword_1"": ""Cybersecurity"", ""keyword_2"": ""SIEM"", ""keyword_3"": ""Botnet Takedowns""}"
"{""generated_title"": ""Simplifying ETL with DLT: A Game-Changer for Data Teams"", ""keyword_1"": ""DLT"", ""keyword_2"": ""ETL"", ""keyword_3"": ""Data Pipelines""}"
"{""generated_title"": ""Data, AI, and a Dash of Humor: Josue's Market Insights"", ""keyword_1"": ""Data Landscape"", ""keyword_2"": ""AI Trends"", ""keyword_3"": ""Market Insights""}"
"{""generated_title"": ""Revolutionizing Banking Data: SMBC's GenAI-Powered Cloud Transformation"", ""keyword_1"": ""GenAI"", ""keyword_2"": ""Cloud Data Platform"", ""keyword_3"": ""Azure Databricks""}"
"{""generated_title"": ""Effortless AI Deployment: Scaling Models with Mosaic AI"", ""keyword_1"": ""AI Deployment"", ""keyword_2"": ""Model Serving"", ""keyword_3"": ""Generative AI""}"
"{""generated_title"": ""Revolutionizing Text Classification: Introducing AT&T AutoClassify"", ""keyword_1"": ""Natural Language Processing"", ""keyword_2"": ""Binary Classification"", ""keyword_3"": ""Automated Machine Learning""}"
"{""generated_title"": ""Unlocking Insights from Unstructured Data with Databricks AI Functions"", ""keyword_1"": ""Databricks"", ""keyword_2"": ""AI Functions"", ""keyword_3"": ""Unstructured Data""}"


Load the generated data into pandas, and expand the struct column to individual columns.

In [0]:

loaded_titles = spark.sql("SELECT * FROM generated_titles_keywords").toPandas()
display(loaded_titles)

session_desc,extracted_info
"In Bruce’s career in cyber warfare and enterprise cybersecurity, he worked on many of the highest profile botnet and nation state takedowns in history. He also helped build the tech in one of the world’s most advanced SOCs. Bruce will explain what he learned from that experience and why it prompted him to leave early retirement, sell his beloved sports car and co-found ziggiz. We all know there’s more data than ever. Anyone close to cybersecurity also knows that SIEMs, typically at the center of enterprise cybersecurity operations, have become too expensive even at the highest levels of government and Fortune 100s.","{""generated_title"": ""From Botnet Battles to Cybersecurity Revolution: Lessons from the Frontlines"", ""keyword_1"": ""Cybersecurity"", ""keyword_2"": ""SIEM"", ""keyword_3"": ""Botnet Takedowns""}"
"DLT 's home page says, “It’s a declarative ETL framework (...) that helps data teams simplify streaming and batch ETL cost-effectively. Simply define the transformations to perform on your data and let DLT pipelines automatically manage task orchestration, cluster management, monitoring, data quality and error handling. This talk aims to show you how DLT saved me a lot of trouble while on a tight delivery schedule. I’ll show you why the DLT headline is correct. In other words, I hope I will convince you to consider the DLT framework for your next ETL project. I found over 10 reasons why investing in DLT for your next project is worth your time. I will discuss the foundational concepts (Spark SQL and Structured Streaming, Delta Lake) and more importantly, how they paved the way for DLT. The talk is based on my recent experience with two successful projects, which have done very well from their humble beginnings and were so much fun to be part of.","{""generated_title"": ""Simplifying ETL with DLT: A Game-Changer for Data Teams"", ""keyword_1"": ""DLT"", ""keyword_2"": ""ETL"", ""keyword_3"": ""Data Pipelines""}"
"Josue is well known for his practical perspectives on the data and AI landscape. We'll talk about what he is seeing in the market, his take on product feature updates, and some humor mixed in.","{""generated_title"": ""Data, AI, and a Dash of Humor: Josue's Market Insights"", ""keyword_1"": ""Data Landscape"", ""keyword_2"": ""AI Trends"", ""keyword_3"": ""Market Insights""}"
"SMBC, a major Japanese multinational financial services institution, has embarked on an initiative to build a GenAI-powered, modern and well-governed cloud data platform on Azure/Databricks. This initiative aims to build an enterprise data foundation encompassing loans, deposits, securities, derivatives, and other data domains. Its primary goals are: To decommission legacy data platforms and reduce data sprawl by migrating 20+ core banking systems to a multi-tenant Azure Databricks architecture To leverage Databrick’s delta-share capabilities to address SMBC’s unique global footprint and data sharing needs To govern data by design using Unity Catalog To achieve global adoption of the frameworks, accelerators, architecture and tool stack to support similar implementations across EMEA Deloitte and SMBC leveraged the Brickbuilder asset “Data as a Service for Banking” to accelerate this highly strategic transformation.","{""generated_title"": ""Revolutionizing Banking Data: SMBC's GenAI-Powered Cloud Transformation"", ""keyword_1"": ""Cloud Data Platform"", ""keyword_2"": ""GenAI"", ""keyword_3"": ""Azure Databricks""}"
"Deploying AI in production is getting more complex — with different model types, tighter timelines, and growing infrastructure demands. In this session, we’ll walk through how Mosaic AI Model Serving helps teams deploy and scale both traditional ML and generative AI models efficiently, with built-in monitoring and governance.We’ll also hear from Skyscanner on how they’ve integrated AI into their products, scaled to 100+ production endpoints, and built the processes and team structures to suppor

In [0]:
loaded_complete = pd.concat([
  loaded_titles.drop(columns="extracted_info"),
  loaded_titles["extracted_info"].transform(json.loads).apply(pd.Series)],
axis=1)
display(loaded_complete)

session_desc,generated_title,keyword_1,keyword_2,keyword_3
"In Bruce’s career in cyber warfare and enterprise cybersecurity, he worked on many of the highest profile botnet and nation state takedowns in history. He also helped build the tech in one of the world’s most advanced SOCs. Bruce will explain what he learned from that experience and why it prompted him to leave early retirement, sell his beloved sports car and co-found ziggiz. We all know there’s more data than ever. Anyone close to cybersecurity also knows that SIEMs, typically at the center of enterprise cybersecurity operations, have become too expensive even at the highest levels of government and Fortune 100s.",From Botnet Battles to Cybersecurity Revolution: Lessons from the Frontlines,Cybersecurity,SIEM,Botnet Takedowns
"DLT 's home page says, “It’s a declarative ETL framework (...) that helps data teams simplify streaming and batch ETL cost-effectively. Simply define the transformations to perform on your data and let DLT pipelines automatically manage task orchestration, cluster management, monitoring, data quality and error handling. This talk aims to show you how DLT saved me a lot of trouble while on a tight delivery schedule. I’ll show you why the DLT headline is correct. In other words, I hope I will convince you to consider the DLT framework for your next ETL project. I found over 10 reasons why investing in DLT for your next project is worth your time. I will discuss the foundational concepts (Spark SQL and Structured Streaming, Delta Lake) and more importantly, how they paved the way for DLT. The talk is based on my recent experience with two successful projects, which have done very well from their humble beginnings and were so much fun to be part of.",Simplifying ETL with DLT: A Game-Changer for Data Teams,DLT,ETL,Data Pipelines
"Josue is well known for his practical perspectives on the data and AI landscape. We'll talk about what he is seeing in the market, his take on product feature updates, and some humor mixed in.","Data, AI, and a Dash of Humor: Josue's Market Insights",Data Landscape,AI Trends,Market Insights
"SMBC, a major Japanese multinational financial services institution, has embarked on an initiative to build a GenAI-powered, modern and well-governed cloud data platform on Azure/Databricks. This initiative aims to build an enterprise data foundation encompassing loans, deposits, securities, derivatives, and other data domains. Its primary goals are: To decommission legacy data platforms and reduce data sprawl by migrating 20+ core banking systems to a multi-tenant Azure Databricks architecture To leverage Databrick’s delta-share capabilities to address SMBC’s unique global footprint and data sharing needs To govern data by design using Unity Catalog To achieve global adoption of the frameworks, accelerators, architecture and tool stack to support similar implementations across EMEA Deloitte and SMBC leveraged the Brickbuilder asset “Data as a Service for Banking” to accelerate this highly strategic transformation.",Revolutionizing Banking Data: SMBC's GenAI-Powered Cloud Transformation,Cloud Data Platform,GenAI,Azure Databricks
"Deploying AI in production is getting more complex — with different model types, tighter timelines, and growing infrastructure demands. In this session, we’ll walk through how Mosaic AI Model Serving helps teams deploy and scale both traditional ML and generative AI models efficiently, with built-in monitoring and governance.We’ll also hear from Skyscanner on how they’ve integrated AI into their products, scaled to 100+ production endpoints, and built the processes and team structures to support AI at scale. Key Takeaways: How Skyscanner ships and operates AI in real-world products How to deploy and scale a variety of models with low latency and minimal overhead Building compound AI systems using models, feature stores, and vector search Monitoring, debugging, and governing production workloads",Effortless AI D

Evaluate our using custom metric!

First, we define a custom [prompt-based scorer](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/create-prompt-judge) to evaluate how catchy our titles are.

In [0]:
from mlflow.genai.judges import custom_prompt_judge
from typing import List, Dict, Any, cast

title_catchiness_prompt = """
Title Catchiness is a measure of how catchy a title is for the Data and AI Summit
Rate the catchness of title using:
  - Emotional appeal
  - Curiosity spark
  - Memorability
  - Brevity

Rubric:
[[Not_catchy]]: score 0, the title is not catchy for the Data and AI Summit event
[[Pretty_catchy]]: score 1, the title is pretty catchy for the Data and AI Summit event
[[Very_catchy]]: score 2, the title is very catchy for the Data and AI Summit event

Title: {{title}}
"""

In [0]:
# Create a data structure that migrates from mlflow 2 to mlflow 3
eval_data = [
    {
        "inputs": {"session_desc": row["session_desc"]},
        "outputs": {"generated_title": row["generated_title"]}
    }
    for _, row in loaded_complete.iterrows()
]

@scorer(aggregations=["mean", "variance"])
def title_catchiness(inputs: Dict[Any, Any], outputs: Dict[Any, Any]):
    title = outputs.get("generated_title", "")
    judge = custom_prompt_judge(
        name="title_catchiness",
        prompt_template=title_catchiness_prompt,
        numeric_values={"Not_catchy": 0.0, "Pretty_catchy": 1.0, "Very_catchy": 2.0}
    )
    return judge(title=title)

with mlflow.start_run() as run: # Create a new mlflow run
  results = mlflow.genai.evaluate(
    data = eval_data,
    predict_fn=None,
    scorers=[title_catchiness],
  )
  # Calculate and log the metrics
  mlflow.log_metrics( 
    results.metrics
  )
print(results.metrics)

Evaluating:   0%|          | 0/7 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

{'agent/latency_seconds/mean': 0.00014285714285714287, 'title_catchiness/mean': 1.1428571428571428}


[Trace(trace_id=tr-1171b65cd309db61b223983b0b5e532e), Trace(trace_id=tr-08ff657af45b43b3b9d1782447751803), Trace(trace_id=tr-227d062d7c632ea2ac3dec85cde6f10a), Trace(trace_id=tr-bc3ace7b40764833ece9d26bfb61d56e), Trace(trace_id=tr-eb511cc907c98122a3f6b863f5ba5bd8), Trace(trace_id=tr-071f7fcce39782b42f33525bef8caabb), Trace(trace_id=tr-178363c750c8a98f6c683347a6217a77)]

# Use Case 3: Rewrite Abstract for Personas
_Why?_ We want to market each of the sessions to Data Engineers. We should rewrite the abstract in a way which will better appeal to them.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW session_summaries_data_engineer AS
SELECT 
  session_desc, 
  ai_gen("""Rewrite the following abstract to appeal to Data Engineers. Abstract: """ || session_desc) as tailored_abstract
FROM dais_sessions

In [0]:
sessions_de = spark.sql("SELECT session_desc, tailored_abstract FROM session_summaries_data_engineer")
display(sessions_de)

session_desc,tailored_abstract
"In Bruce’s career in cyber warfare and enterprise cybersecurity, he worked on many of the highest profile botnet and nation state takedowns in history. He also helped build the tech in one of the world’s most advanced SOCs. Bruce will explain what he learned from that experience and why it prompted him to leave early retirement, sell his beloved sports car and co-found ziggiz. We all know there’s more data than ever. Anyone close to cybersecurity also knows that SIEMs, typically at the center of enterprise cybersecurity operations, have become too expensive even at the highest levels of government and Fortune 100s.","Here's a rewritten abstract that appeals to Data Engineers: ""Join us as Bruce, a seasoned expert in cyber warfare and enterprise cybersecurity, shares his insights on the challenges of handling massive amounts of data in cybersecurity operations. With a background in leading high-profile botnet and nation-state takedowns, Bruce has witnessed firsthand the limitations of traditional Security Information and Event Management (SIEM) systems in processing and analyzing the ever-increasing volumes of data. As a data engineer, you're likely familiar with the scalability and cost challenges of implementing effective cybersecurity solutions. Bruce will discuss how his experience in building advanced Security Operations Centers (SOCs) and working with large datasets led him to co-found ziggiz, a company that aims to revolutionize the way we approach cybersecurity data management. In this talk, Bruce will delve into the technical challenges of handling cybersecurity data at scale, the limitations of traditional SIEM systems, and the opportunities for innovation in this space. If you're interested in learning about the intersection of data engineering and cybersecurity, and how to build more efficient and cost-effective solutions, this talk is for you."""
"DLT 's home page says, “It’s a declarative ETL framework (...) that helps data teams simplify streaming and batch ETL cost-effectively. Simply define the transformations to perform on your data and let DLT pipelines automatically manage task orchestration, cluster management, monitoring, data quality and error handling. This talk aims to show you how DLT saved me a lot of trouble while on a tight delivery schedule. I’ll show you why the DLT headline is correct. In other words, I hope I will convince you to consider the DLT framework for your next ETL project. I found over 10 reasons why investing in DLT for your next project is worth your time. I will discuss the foundational concepts (Spark SQL and Structured Streaming, Delta Lake) and more importantly, how they paved the way for DLT. The talk is based on my recent experience with two successful projects, which have done very well from their humble beginnings and were so much fun to be part of.","Here's a rewritten abstract that appeals to Data Engineers: **Simplifying ETL Workflows with DLT: A Data Engineer's Game-Changer** As Data Engineers, we're constantly looking for ways to streamline our ETL processes, reduce costs, and improve data quality. DLT, a declarative ETL framework, promises to do just that by automating task orchestration, cluster management, monitoring, and error handling. But does it live up to the hype? In this talk, I'll share my hands-on experience with DLT, highlighting over 10 key benefits that made it a crucial component of my recent projects. I'll dive into the foundational concepts that power DLT, including Spark SQL, Structured Streaming, and Delta Lake, and demonstrate how they come together to simplify streaming and batch ETL workflows. Through real-world examples from two successful projects, I'll show you how DLT can help you: * Simplify ETL pipeline management * Reduce costs and improve scalability * Enhance data quality and error handling * Increase productivity and efficiency If you're looking for a reliable and cost-effective ETL solution, join me to lea

Let's perform the same query with a reasoning model and then we can benchmark and compare

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW session_summaries_data_engineer_reasoning AS
SELECT  
session_desc,
ai_query(
    "databricks-claude-3-7-sonnet",
    """Rewrite the following abstract to appeal to Data Engineers. Abstract: """ || session_desc,
    modelParameters =>
      named_struct(
        "max_tokens", 1025, 
        "thinking", named_struct(
          "type", "enabled", 
          "budget_tokens", 1024
        ),
        "temperature", 0.0
      )) as tailored_abstract
FROM dais_sessions

In [0]:
sessions_de_tailored = spark.sql("SELECT session_desc, tailored_abstract FROM session_summaries_data_engineer_reasoning")
display(sessions_de_tailored)

session_desc,tailored_abstract
"In Bruce’s career in cyber warfare and enterprise cybersecurity, he worked on many of the highest profile botnet and nation state takedowns in history. He also helped build the tech in one of the world’s most advanced SOCs. Bruce will explain what he learned from that experience and why it prompted him to leave early retirement, sell his beloved sports car and co-found ziggiz. We all know there’s more data than ever. Anyone close to cybersecurity also knows that SIEMs, typically at the center of enterprise cybersecurity operations, have become too expensive even at the highest levels of government and Fortune 100s.","# From Cyber Warfare to Enterprise Data Solutions: Building Scalable Security Infrastructure During his career in cyber warfare and enterprise security, Bruce architected data pipelines and processing systems behind some of history's most significant botnet and nation state takedowns. His work involved designing distributed data collection frameworks and real-time analytics platforms that formed the backbone of one of the world's most sophisticated Security Operations Centers. Bruce will share his data engineering insights from scaling these high-throughput security systems and explain how these challenges drove him out of retirement to co-found ziggiz, even requiring him to part with his prized sports car to fund the venture. In an era where data volumes are exploding exponentially, he'll address the critical infrastructure problems facing security teams. Current SIEM platforms—which should function as optimal data lakes and processing centers for security operations—have become prohibitively expensive and inefficient at scale, creating bottlenecks even for organizations with enterprise-level resources like government agencies and Fortune 100 companies. Learn how modern data engineering approaches can revolutionize security analytics infrastructure."
"DLT 's home page says, “It’s a declarative ETL framework (...) that helps data teams simplify streaming and batch ETL cost-effectively. Simply define the transformations to perform on your data and let DLT pipelines automatically manage task orchestration, cluster management, monitoring, data quality and error handling. This talk aims to show you how DLT saved me a lot of trouble while on a tight delivery schedule. I’ll show you why the DLT headline is correct. In other words, I hope I will convince you to consider the DLT framework for your next ETL project. I found over 10 reasons why investing in DLT for your next project is worth your time. I will discuss the foundational concepts (Spark SQL and Structured Streaming, Delta Lake) and more importantly, how they paved the way for DLT. The talk is based on my recent experience with two successful projects, which have done very well from their humble beginnings and were so much fun to be part of.","# Streamlining ETL Operations with Delta Live Tables: A Data Engineer's Perspective **Abstract:** As data engineers, we're constantly battling complex pipeline orchestration, maintenance overhead, and reliability issues in our ETL workflows. Delta Live Tables (DLT) addresses these pain points head-on with its declarative framework that dramatically simplifies both streaming and batch ETL processes while optimizing resource utilization. This technical deep dive will demonstrate how DLT eliminated critical bottlenecks in production environments under tight deadlines. You'll see firsthand how writing simple transformation definitions—rather than managing complex orchestration code—can accelerate your development cycle and reduce operational overhead. I'll present 10+ concrete technical advantages that make DLT a compelling choice for your infrastructure, including automated dependency management, built-in data quality enforcement, and simplified error handling that would otherwise require extensive custom coding. The presentation will examine the technical architecture connecting Spark SQL, Str

In [0]:
data_engineer_custom_prompt = """
Data Engineer relevancy reviews a provided abstract and scores how relevant it is to a typical Data Engineer. 
Data Engineer Priorities include SQL/database expertise, Python programming, Data modeling, ETL pipeline building, Cloud/big data tools.

categorize the provided abstract using Rubric with scale from 0 to 5:

Rubric:
[[Not_relevant]]: the session is not for Data Engineers and unrelated data engineer priorities. (score of 0)
[[relevant]]: the session is appealing to data engineers and is relevant Data Engineer priorities but may not fully focus on the priorities (score of 3)
[[Golden_standard]]: This is the golden standard for a Data Engineer sessions. the session focus on data engineer proirities. (score of 5)

abstract: {{tailored_abstract}}
"""

Test the `ai_gen` tailored abstracts.

In [0]:
# Create a data structure that migrates from mlflow 2 to mlflow 3
eval_data = [
    {
        "inputs": {"session_desc": row["session_desc"]},
        "outputs": {"tailored_abstract": row["tailored_abstract"]}
    }
    for _, row in sessions_de.toPandas().iterrows()
]

eval_data_tailored = [
    {
        "inputs": {"session_desc": row["session_desc"]},
        "outputs": {"tailored_abstract": row["tailored_abstract"]}
    }
    for _, row in sessions_de_tailored.toPandas().iterrows()
]

In [0]:
@scorer(aggregations=["mean", "variance"])
def data_engineer_relevancy(inputs: Dict[Any, Any], outputs: Dict[Any, Any]):
    tailored_abstract = outputs.get("tailored_abstract", "")
    judge = custom_prompt_judge(
        name="data_engineer_relevancy",
        prompt_template=data_engineer_custom_prompt,
        numeric_values={"Not_relevant": 0.0, "relevant": 3.0, "Golden_standard": 5.0}
    )
    return judge(tailored_abstract=tailored_abstract)
  

with mlflow.start_run() as run: # Create a new mlflow run
  results = mlflow.genai.evaluate(
    data = eval_data,
    predict_fn=None,
    scorers=[data_engineer_relevancy]
  )
  # Calculate and log the metrics
  mlflow.log_metrics( 
    results.metrics
  )
print(results.metrics)

Evaluating:   0%|          | 0/7 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

{'agent/latency_seconds/mean': 0.0, 'data_engineer_relevancy/mean': 4.333333333333333}


[Trace(trace_id=tr-40cd07a0256178d2f2e387d629e0ed2f), Trace(trace_id=tr-639aefb4197a5b74d1c97a7b4cf96e8d), Trace(trace_id=tr-3313ebd2959af215774f1ad6719347ab), Trace(trace_id=tr-59af39051d69e3fb03d499ee548d73f4), Trace(trace_id=tr-04d73063306a677f3eab6463167c2f65), Trace(trace_id=tr-a8c7a614a2297c3aa1ade71b0d9b72fa), Trace(trace_id=tr-e70a162e96398c856ea8cb1275217cff)]

Test the reasoning abstracts.

In [0]:
with mlflow.start_run() as run: # Create a new mlflow run
  results = mlflow.genai.evaluate(
    data = eval_data_tailored,
    predict_fn=None,
    scorers=[data_engineer_relevancy]
  )
  # Calculate and log the metrics
  mlflow.log_metrics( 
    results.metrics
  )
print(results.metrics)

Evaluating:   0%|          | 0/7 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

{'agent/latency_seconds/mean': 0.0, 'data_engineer_relevancy/mean': 5.0}


[Trace(trace_id=tr-694e280d182be347ca856587e52981c0), Trace(trace_id=tr-807d2857fd1ab4642d38e5f8c00e03be), Trace(trace_id=tr-3040d305218e76b205ce2fd74d46c63b), Trace(trace_id=tr-7937ecfc92db787e7a69ad9861143fc2), Trace(trace_id=tr-1468b56f5498f37b4112f683fcb71a5f), Trace(trace_id=tr-9f9343780369fc703d41b41252fe590a), Trace(trace_id=tr-77eba5485b253bf8f2c56d34a481f8e6)]

With claude sonnect 3.7 thinking mode, the abstract indeed improved based on the custome metrics of `data_engineering_relevancy`